# 10 — BES vs CAA on current Llama and Qwen models, with the full rank sweep

Runs one protocol, unchanged, on each model: question-grouped splits, BES (every rank from 1 to 256 plus a learned eraser) and CAA (layer, span and multiplier grid) both chosen on validation, CAA matched to BES's sycophancy level, then a single pass over the held-out test questions with MMLU and the factual control.

In [ ]:
# Environment: on Colab clone the repo, install requirements and a recent transformers (Qwen3.5 support), mount Drive; then build the pairs
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/chase-mayer/syco-erase.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/syco-erase").exists():
        subprocess.run(["git", "clone", "-q", REPO, "/content/syco-erase"], check=True)
    os.chdir("/content/syco-erase")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention", "bitsandbytes"], check=False)  # Qwen3.5 kernels; 4-bit loading
    from google.colab import drive
    drive.mount("/content/drive")
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
assert "CHAT_TEMPLATE_KWARGS" in Path("src/bes.py").read_text(), "stale clone: push the updated repo, delete /content/syco-erase and rerun"
subprocess.run(["bash", "scripts/download_data.sh"], check=True, capture_output=True)
print(subprocess.run([sys.executable, "-m", "src.data", "--dataset", "all"],
                     check=True, capture_output=True, text=True).stdout)

In [ ]:
# Hugging Face login for the gated Llama weights (skipped in offline smoke tests)
if os.environ.get("BES_SMOKE") != "1":
    from huggingface_hub import login
    login()

## 1. Models and protocol settings

Every model gets the same splits, grids and selection rules; only the prompt format follows each model (its own chat template, except Qwen2.5, which keeps the raw format and fp32 precision of notebooks 02–05).

In [ ]:
# Models to run and the shared protocol settings (reduced automatically in smoke tests)
import gc, json, time
import numpy as np
import pandas as pd
import torch
import transformers
from src import bes, experiment as E, plots, stats

SMOKE = os.environ.get("BES_SMOKE") == "1"
MODELS = [
    {"tag": "qwen25_1p5b", "name": "Qwen/Qwen2.5-1.5B-Instruct", "chat": False, "dtype": "fp32", "role": "original Qwen; adds the ranks above 8 and the learned eraser"},
    {"tag": "qwen35_2b", "name": "Qwen/Qwen3.5-2B", "chat": True, "role": "current Qwen, closest in size to the original"},
    {"tag": "llama32_3b", "name": "meta-llama/Llama-3.2-3B-Instruct", "chat": True, "role": "newest Llama text model under 8B (smaller than 3.1-8B)"},
    {"tag": "llama31_8b", "name": "meta-llama/Llama-3.1-8B-Instruct", "chat": True, "role": "largest current Llama that fits one GPU; Llama-2-7B's size"},
    {"tag": "qwen35_9b", "name": "Qwen/Qwen3.5-9B", "chat": True, "role": "current Qwen at Llama-3.1's size", "optional": True},
    {"tag": "llama2_7b", "name": "meta-llama/Llama-2-7b-chat-hf", "chat": True, "role": "original Llama under this protocol", "optional": True},
    {"tag": "muse_glimmer_30b", "name": "meta-models/Muse-Glimmer-30B", "chat": True, "quant": "auto",
     "role": "current Meta open model (not a Llama); 4-bit unless the GPU has ~80 GB", "optional": True},
]
RUN_OPTIONAL = {"qwen35_9b": False, "llama2_7b": False, "muse_glimmer_30b": False}   # switch on the optional models to run
REDO = False                  # False: skip a model whose results are already saved
if SMOKE:
    MODELS = [{"tag": "smoke_raw", "name": os.environ["BES_MODEL"], "chat": False, "role": "smoke"},
              {"tag": "smoke_chat", "name": os.environ["BES_MODEL"], "chat": True, "role": "smoke"}]

bes.CHAT_TEMPLATE_KWARGS.update({"enable_thinking": False,      # Qwen3.5: answer directly, no reasoning block
                                 "date_string": "26 Jul 2024"})  # Llama-3.1: fixed date in its system header
LAYER_FRACS = np.arange(0.25, 0.95, 0.1)        # the layer grid of notebook 03, as fractions of depth
SCREEN_RANKS = (4, 16, 64)                       # ranks tried at every layer
RANKS = (1, 2, 4, 8, 16, 32, 64, 128, 256)       # full sweep at the chosen layer
STRENGTHS = (0.5, 0.75, 1.0)
LEARN_K = (4, 16)                                # learned erasers
LEARN_STEPS, LEARN_BS, LEARN_LR = (450, 4, 2e-3) if not SMOKE else (6, 4, 2e-3)
ALPHAS = (-0.5, -1.0, -2.0, -4.0, -8.0, -16.0, -32.0)
CAA_SPANS = ("last", "question", "all")
MIN_DROP = 0.03                                  # a setting must lower validation sycophancy by this much to count
N_MMLU, N_FACT = (1000, 500) if not SMOKE else (40, 40)
print("transformers", transformers.__version__, "| Qwen3.5 support:", hasattr(transformers, "Qwen3_5ForCausalLM"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
pd.DataFrame(MODELS)

## 2. Shared pieces

Scoring, the selection rule (lowest sycophancy + KL on validation), direction fitting and the learned eraser, written once and applied to every model.

In [ ]:
# Scoring, selection and model loading
def summ(per):
    return {"syco": float(per["syco"].mean()), "kl": float(per["kl"].mean())}


def val_score(run, items, iv=None, span="all"):
    return summ(run.evaluate(items, iv, span)[1])


def best_by_j(df, base_syco):
    """Lowest sycophancy + KL among settings that lower sycophancy by MIN_DROP (all settings if none do)."""
    ok = df[df.syco <= base_syco - MIN_DROP]
    pool = ok if len(ok) else df
    return pool.assign(j=pool.syco + pool.kl).sort_values("j").iloc[0], bool(len(ok))


def load_runner(spec):
    dtype = spec.get("dtype") or ("bf16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else ("fp16" if torch.cuda.is_available() else "fp32"))
    quant = spec.get("quant")
    if quant == "auto":   # 4-bit on a 40 GB card, full bf16 on an 80 GB card
        big = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 70e9
        quant = None if big else "4bit"
    cfg = E.Config(model=spec["name"], chat=spec["chat"], dtype=dtype, batch_size=16 if not quant else 8, quant=quant)
    run = E.Runner(cfg)
    for p_ in run.model.parameters():
        p_.requires_grad_(False)
    return cfg, run


def layer_grid(n_layers):
    if SMOKE:
        return list(range(1, n_layers))
    return sorted({max(1, int(f * n_layers)) for f in LAYER_FRACS})


def rank_cap(run, n_rows):
    return min(run.d_model // 4, n_rows - 1)

In [ ]:
# BES directions: opinion-minus-neutral differences over the question window, centred SVD on the GPU
def differences(run, items, layers, W):
    ho = run.activations(items.opinion, layers, W)
    hn = run.activations(items.neutral, layers, W)
    return {L: ho[L] - hn[L] for L in layers}


def svd_basis(run, d, k):
    x = d.to(run.device, torch.float32)
    x = x - x.mean(0, keepdim=True)
    _, S, Vh = torch.linalg.svd(x, full_matrices=False)
    k = min(k, Vh.shape[0])
    return Vh[:k].cpu(), (S[:k] ** 2 / (S ** 2).sum()).cpu().numpy()


def project(L, basis, s=1.0, band=0, n_layers=None):
    layers = [L] if band == 0 else E.band_layers(L, band, n_layers)
    return bes.ProjectOut(layers=layers, basis=basis, strength=s)

In [ ]:
# Learned eraser: k orthonormal directions at one layer, trained with the model frozen to minimise KL to the no-opinion answers
class Learned(bes.Intervention):
    def __init__(self, layer, Q):
        self.layers, self.mask, self.Q = [layer], None, Q

    def edit(self, h):
        x = h.float()
        c = torch.einsum("bld,kd->blk", x, self.Q)
        return (x - torch.einsum("blk,kd->bld", c, self.Q)).to(h.dtype)


def train_learned(run, fit, val, L, init, base_syco, tag):
    U = torch.nn.Parameter(init.to(run.device, torch.float32))
    opt = torch.optim.Adam([U], lr=LEARN_LR)
    ids = bes.option_ids(run.tok)
    valid = torch.tensor(bes.letter_mask(fit.options), device=run.device)
    p0_all = torch.tensor(fit.p_neutral, device=run.device, dtype=torch.float32)
    order = np.random.RandomState(0).permutation(len(fit.rows))
    log, best = [], (None, float("inf"))
    for step in range(LEARN_STEPS):
        start = (step * LEARN_BS) % len(order)
        idx = order[start: start + LEARN_BS]
        if len(idx) < 2:
            continue
        Q = torch.linalg.qr(U.T)[0].T
        iv = Learned(L, Q)
        enc = bes._encode(run.tok, [fit.opinion[j] for j in idx], run.device)
        iv.mask = bes.span_mask(enc["input_ids"].shape[1], [fit.spans["question"][j] for j in idx])
        with bes.installed(run.model, iv):
            logits = bes._last_logits(run.model, enc).float()
        p = torch.softmax(logits, -1)[:, ids] * valid[idx]
        p = p / p.sum(1, keepdim=True).clamp(min=1e-9)
        p0 = p0_all[idx]
        loss = torch.where(p0 > 0, p0 * (torch.log(p0.clamp(min=1e-9)) - torch.log(p.clamp(min=1e-9))), torch.zeros_like(p0)).sum(1).mean()
        opt.zero_grad()
        if torch.isfinite(loss):
            loss.backward()
            opt.step()
        if step % 50 == 0 or step == LEARN_STEPS - 1:
            with torch.no_grad():
                Qv = torch.linalg.qr(U.T)[0].T.detach().cpu()
            r = val_score(run, val, project(L, Qv), "question")
            log.append({"model": tag, "k": init.shape[0], "step": step, "train_kl": float(loss.detach()), **r})
            if r["syco"] + r["kl"] < best[1]:
                best = (Qv.clone(), r["syco"] + r["kl"])
    return best[0], pd.DataFrame(log)

In [ ]:
# CAA (Rimsky et al.): mean activation difference at the answer letter, user's answer minus another answer, fitted at many layers in one pass
def caa_vectors(run, items, layers, seed=0):
    rng = __import__("random").Random(seed)
    syco, non = [], []
    for p, r in zip(items.opinion, items.rows):
        syco.append(p + r["matching"])
        non.append(p + rng.choice(r["not_matching_all"]))
    hs, hn = run.activations(syco, layers, 1), run.activations(non, layers, 1)
    return {L: bes.caa_vector(hs[L], hn[L]) for L in layers}


def caa_at_target(grid, base, target):
    """For each (layer, span), walk from no intervention to stronger multipliers and read KL where sycophancy first reaches
    `target`; return the (layer, span, alpha) with the lowest such KL (or the closest miss if none reaches it)."""
    rows = []
    for (L, sp), d in grid.groupby(["layer", "span"]):
        d = d.sort_values("alpha", ascending=False)
        alphas = [0.0, *d.alpha]
        pos, kl = stats.first_crossing([base["syco"], *d.syco], [base["kl"], *d.kl], target)
        alpha = float(np.interp(pos, range(len(alphas)), alphas)) if np.isfinite(pos) else np.nan
        close = d.iloc[(d.syco - target).abs().argmin()]
        rows.append({"layer": L, "span": sp, "alpha": alpha, "kl_at_target": kl, "closest_alpha": close.alpha,
                     "miss": abs(close.syco - target), "closest_kl": close.kl})
    m = pd.DataFrame(rows)
    hit = m.kl_at_target.notna()
    if hit.any():
        r = m[hit].sort_values("kl_at_target").iloc[0]
        return int(r.layer), r.span, float(r.alpha), True
    r = m.sort_values(["miss", "closest_kl"]).iloc[0]
    return int(r.layer), r.span, float(r.closest_alpha), False

## 3. The protocol for one model

Validation chooses everything; the test questions are scored once, at the end, for every method and the matched CAA settings.

In [ ]:
# Full protocol for one model: splits, BES layer/rank/strength search, learned erasers, CAA grid, matching, test, MMLU, factual control
def run_model(spec):
    tag, t0 = spec["tag"], time.time()
    out_test = E.RESULTS / f"m10_{tag}_test.csv"
    if out_test.exists() and not REDO:
        print(f"{tag}: results already saved, skipping (set REDO = True to rerun)")
        return pd.read_csv(E.RESULTS / f"m10_{tag}_summary.csv")
    try:
        cfg, run = load_runner(spec)
    except OSError as exc:   # gated repo without access, or a wrong name
        print(f"{tag}: could not load {spec['name']} — request access on its Hugging Face page, then rerun this cell.\n  {str(exc)[:300]}")
        return None
    nl = run.n_layers
    clock = lambda: f"{(time.time() - t0) / 60:.1f} min"

    # data: the seed-0 question-grouped splits used throughout the project
    splits = E.make_splits(cfg, seed=0)
    fit = run.baselines(run.prepare(splits["fit"]))
    fit_caa = run.prepare(splits["fit_caa"])
    val = run.baselines(run.prepare(splits["val"]))
    test = run.baselines(run.prepare(splits["test"]))
    W = run.window_for(fit, cfg.window)
    iv_ = np.arange(len(val.rows))
    BASE = val_score(run, val)
    ORACLE = float(val.p_neutral[iv_, val.match].mean())
    print(f"{spec['name']} | layers {nl}, d {run.d_model}, {cfg.dtype}{', ' + cfg.quant if cfg.quant else ''}, chat {cfg.chat} | val: none {BASE['syco']:.3f} (KL {BASE['kl']:.3f}), "
          f"no-opinion {ORACLE:.3f} | window {W}")
    print("  example prompt tail:", repr(val.opinion[0][-120:]))
    if BASE["syco"] - ORACLE < MIN_DROP:
        print("  warning: this model is barely sycophantic on validation; differences between methods will be small")

    # BES step 1: every layer in the grid at three ranks
    layers = layer_grid(nl)
    cap = rank_cap(run, len(fit.rows) * W)
    diffs = differences(run, fit, layers, W)
    bases = {L: svd_basis(run, diffs[L], max(r for r in RANKS if r <= cap))[0] for L in layers}
    del diffs
    screen = pd.DataFrame([{"model": tag, "layer": L, "rank": k, **val_score(run, val, project(L, bases[L][:k]), "question")}
                           for L in layers for k in SCREEN_RANKS if k <= cap])
    E.save(screen, f"m10_{tag}_bes_layer_screen")
    LB = int(best_by_j(screen, BASE["syco"])[0].layer)
    print(f"  BES layer screen done ({clock()}): layer {LB}")

    # BES step 2: full rank x strength sweep at that layer, then a +/-1 layer band for the three best settings
    B = bases[LB]
    sweep = pd.DataFrame([{"model": tag, "layer": LB, "rank": k, "strength": s, "band": 0,
                           **val_score(run, val, project(LB, B[:k], s), "question")} for k in RANKS if k <= cap for s in STRENGTHS])
    top = sweep.assign(j=sweep.syco + sweep.kl).sort_values("j").head(3)
    band = pd.DataFrame([{"model": tag, "layer": LB, "rank": int(r["rank"]), "strength": r.strength, "band": 1,
                          **val_score(run, val, project(LB, B[:int(r["rank"])], r.strength, 1, nl), "question")} for _, r in top.iterrows()])
    sweep = pd.concat([sweep, band], ignore_index=True)
    E.save(sweep, f"m10_{tag}_bes_rank_sweep")
    svd_pick, svd_ok = best_by_j(sweep, BASE["syco"])
    print(f"  BES rank sweep done ({clock()}): rank {int(svd_pick['rank'])}, strength {svd_pick.strength}, band {int(svd_pick.band)} "
          f"-> val syco {svd_pick.syco:.3f}, KL {svd_pick.kl:.3f}" + ("" if svd_ok else " [no setting met the minimum drop]"))

    # BES step 3: learned erasers, initialised from the top SVD directions, checkpoint and strength chosen on validation
    learned, logs = {}, []
    for K in LEARN_K:
        if K > cap:
            continue
        Q, lg = train_learned(run, fit, val, LB, B[:K].clone(), BASE["syco"], tag)
        logs.append(lg)
        ls = pd.DataFrame([{"model": tag, "k": K, "strength": s, **val_score(run, val, project(LB, Q, s), "question")} for s in STRENGTHS])
        pick_l, ok_l = best_by_j(ls, BASE["syco"])
        learned[K] = {"Q": Q, "strength": float(pick_l.strength), "ok": ok_l, "syco": pick_l.syco, "kl": pick_l.kl, "table": ls}
        torch.cuda.empty_cache()
    if logs:
        E.save(pd.concat(logs), f"m10_{tag}_learned_log")
        E.save(pd.concat([v["table"] for v in learned.values()]), f"m10_{tag}_learned_strength")
    print(f"  learned erasers done ({clock()}): " + ", ".join(f"k{K} s{v['strength']} val syco {v['syco']:.3f} KL {v['kl']:.3f}" for K, v in learned.items()))

    # CAA steelman: 1,000 fitting items, every layer x span x multiplier on validation
    vecs = caa_vectors(run, fit_caa, layers)
    grid = pd.DataFrame([{"model": tag, "layer": L, "span": sp, "alpha": a, "vec_norm": float(vecs[L].norm()),
                          **val_score(run, val, bes.AddVector(layers=[L], vector=vecs[L], alpha=a), sp)}
                         for L in layers for sp in CAA_SPANS for a in ALPHAS])
    E.save(grid, f"m10_{tag}_caa_grid")
    caa_own, _ = best_by_j(grid, BASE["syco"])
    print(f"  CAA grid done ({clock()}): best on its own terms layer {int(caa_own.layer)}, {caa_own.span}, alpha {caa_own.alpha:g}")

    # BES candidates for test, and CAA matched to each one's validation sycophancy
    cands = [("none", None, "all", "reference")]
    cands += [(f"BES rank {k}", project(LB, B[:k], 1.0), "question", "rank curve") for k in RANKS if k <= cap]
    heads = [("BES selected (SVD)", project(LB, B[:int(svd_pick["rank"])], float(svd_pick.strength), int(svd_pick.band), nl), svd_pick.syco)]
    heads += [(f"BES learned k{K}", project(LB, v["Q"], v["strength"]), v["syco"]) for K, v in learned.items()]
    matched_rows = []
    for name, iv, target in heads:
        cands.append((name, iv, "question", "headline"))
        L, sp, a, hit = caa_at_target(grid, BASE, target)
        cands.append((f"CAA matched to {name}", bes.AddVector(layers=[L], vector=vecs[L], alpha=a), sp, "headline"))
        matched_rows.append({"model": tag, "bes": name, "target_val_syco": target, "caa_layer": L, "caa_span": sp, "caa_alpha": a, "reached": hit})
    cands.append(("CAA best on its own terms", bes.AddVector(layers=[int(caa_own.layer)], vector=vecs[int(caa_own.layer)], alpha=float(caa_own.alpha)),
                  caa_own.span, "headline"))
    m0 = matched_rows[0]
    cands += [(f"CAA curve a={a:g}", bes.AddVector(layers=[m0["caa_layer"]], vector=vecs[m0["caa_layer"]], alpha=a), m0["caa_span"], "CAA curve")
              for a in ALPHAS]
    E.save(pd.DataFrame(matched_rows), f"m10_{tag}_caa_matched")

    # Test: every candidate once on the held-out questions, cluster-bootstrap intervals, paired tests vs none and vs matched CAA
    per = {name: run.evaluate(test, iv, span)[1] for name, iv, span, _ in cands}
    per["prompt instruction"] = run.evaluate(test, system=E.IGNORE_OPINION_PROMPT)[1]
    kinds = {name: kind for name, _, _, kind in cands} | {"prompt instruction": "reference"}
    rows = []
    for name, r in per.items():
        s_ci, k_ci = stats.cluster_bootstrap(r["syco"], test.qid), stats.cluster_bootstrap(r["kl"], test.qid)
        row = {"model": tag, "method": name, "kind": kinds[name], "syco": s_ci[0], "syco_lo": s_ci[1], "syco_hi": s_ci[2],
               "kl": k_ci[0], "kl_lo": k_ci[1], "kl_hi": k_ci[2]}
        refs = [("none", "vs_none")] + ([(f"CAA matched to {name}", "vs_matched_caa")] if f"CAA matched to {name}" in per else [])
        for ref, suffix in refs:
            if name != ref:
                for metric in ("syco", "kl"):
                    d = stats.paired_cluster_bootstrap(r[metric], per[ref][metric], test.qid)
                    row[f"d_{metric}_{suffix}"], row[f"p_{metric}_{suffix}"] = d["diff"], d["p"]
        rows.append(row)
    it_ = np.arange(len(test.rows))
    rows.append({"model": tag, "method": "no-opinion reference", "kind": "reference", "syco": float(test.p_neutral[it_, test.match].mean()), "kl": 0.0})
    test_df = pd.DataFrame(rows)
    E.save(test_df, f"m10_{tag}_test")
    print(f"  test done ({clock()})")

    # MMLU on the headline methods, with exact McNemar tests vs none and BES vs its matched CAA
    mm = E.load_mmlu(N_MMLU)
    gold = np.array([r["answer"] for r in mm])
    prompts = [bes.format_prompt(run.tok, r["text"], cfg.chat) for r in mm]
    heads_all = [c for c in cands if c[3] in ("reference", "headline")]
    correct = {name: run.probs(prompts, [["A", "B", "C", "D"]] * len(mm), iv=iv, spans=E.plain_spans(len(mm), span)).argmax(1) == gold
               for name, iv, span, _ in heads_all}
    mm_rows = []
    for name, c in correct.items():
        row = {"model": tag, "method": name, "mmlu_acc": float(c.mean())}
        if name != "none":
            row["p_vs_none"] = stats.mcnemar_exact(c, correct["none"])["p"]
        if f"CAA matched to {name}" in correct:
            row["mmlu_acc_matched_caa"] = float(correct[f"CAA matched to {name}"].mean())
            row["p_vs_matched_caa"] = stats.mcnemar_exact(c, correct[f"CAA matched to {name}"])["p"]
        mm_rows.append(row)
    mmlu_df = pd.DataFrame(mm_rows)
    E.save(mmlu_df, f"m10_{tag}_mmlu")

    # Factual control: does the model still follow a user who is right, and resist one who is wrong?
    fr = E.factual_rows(N_FACT)
    f_gold, f_opts = np.array([r["gold"] for r in fr]), [r["options"] for r in fr]
    fp = {c: [bes.format_prompt(run.tok, r[k], cfg.chat) for r in fr]
          for c, k in (("neutral", "neutral_prompt"), ("user_correct", "user_correct"), ("user_incorrect", "user_incorrect"))}
    f_spans = {"neutral": {sp: E.plain_spans(len(fr), sp) for sp in ("all", "question", "last", "biography")}}
    for c in ("user_correct", "user_incorrect"):
        f_spans[c] = E.make_spans(run.tok, fp[c], fp["neutral"])
    p_ref = run.probs(fp["neutral"], f_opts)
    f_rows = []
    for name, iv, span, _ in heads_all:
        row = {"model": tag, "method": name}
        for c in fp:
            p = run.probs(fp[c], f_opts, iv=iv, spans=f_spans[c][span])
            row[f"acc_{c}"] = float((p.argmax(1) == f_gold).mean())
            row[f"kl_{c}"] = float(bes.kl_rows(p_ref, p).mean())
        f_rows.append(row)
    fact_df = pd.DataFrame(f_rows)
    E.save(fact_df, f"m10_{tag}_factual_control")

    # One-row summary per model for the comparison across models
    t = test_df.set_index("method")
    a = mmlu_df.set_index("method").mmlu_acc
    srow = {"model": tag, "hf_name": spec["name"], "layers": nl, "chat": cfg.chat, "bes_layer": LB,
            "none_syco": t.loc["none", "syco"], "none_kl": t.loc["none", "kl"], "no_opinion_syco": t.loc["no-opinion reference", "syco"],
            "mmlu_none": a["none"], "minutes": round((time.time() - t0) / 60, 1)}
    for name, _, _ in heads:
        key = "svd" if "SVD" in name else name.split()[-1]
        mname = f"CAA matched to {name}"
        srow |= {f"{key}_desc": name if key != "svd" else f"rank {int(svd_pick['rank'])} s{svd_pick.strength:g} band {int(svd_pick.band)}",
                 f"{key}_syco": t.loc[name, "syco"], f"{key}_kl": t.loc[name, "kl"], f"{key}_mmlu": a[name],
                 f"{key}_caa_syco": t.loc[mname, "syco"], f"{key}_caa_kl": t.loc[mname, "kl"], f"{key}_caa_mmlu": a[mname],
                 f"{key}_p_kl_vs_caa": t.loc[name, "p_kl_vs_matched_caa"], f"{key}_p_syco_vs_caa": t.loc[name, "p_syco_vs_matched_caa"]}
    summary = pd.DataFrame([srow])
    E.save(summary, f"m10_{tag}_summary")
    print(f"{tag}: finished in {clock()}")

    # Figure: test frontier with the BES rank curve, the learned erasers, the matched CAA curve and the references
    fig, ax = plots.subplots(figsize=(7.2, 4.6))
    rc = test_df[test_df.kind == "rank curve"].copy()
    rc["k"] = rc.method.str.split().str[-1].astype(int)
    rc = rc.sort_values("k")
    ax.plot(rc.syco, rc.kl, "-o", color=plots.COLORS["BES"], label=f"BES rank 1–{rc.k.max()} (layer {LB}, s=1)")
    for _, r in rc.iterrows():
        ax.annotate(f"k{r.k}", (r.syco, r.kl), textcoords="offset points", xytext=(4, 4), fontsize=7)
    cc = test_df[test_df.kind == "CAA curve"].sort_values("syco")
    ax.plot(cc.syco, cc.kl, "--o", color=plots.COLORS["CAA"], label=f"CAA layer {m0['caa_layer']}, {m0['caa_span']}")
    for (_, r), mk in zip(test_df[test_df.kind == "headline"].iterrows(), ["s", "^", "D", "v", "P", "X", "*"]):
        col = plots.COLORS["CAA"] if r.method.startswith("CAA") else plots.COLORS["BES"]
        ax.errorbar([r.syco], [r.kl], xerr=[[r.syco - r.syco_lo], [r.syco_hi - r.syco]], yerr=[[r.kl - r.kl_lo], [r.kl_hi - r.kl]],
                    fmt=mk, ms=7, capsize=2, color=col, label=r.method)
    ax.scatter([t.loc["none", "syco"]], [t.loc["none", "kl"]], marker="s", s=60, color=plots.COLORS["none"], label="none", zorder=4)
    ax.axvline(t.loc["no-opinion reference", "syco"], color=plots.COLORS["oracle"], ls="--", lw=1, label="no-opinion level")
    ax.set_xlabel("P(user's answer)  ← less sycophantic"); ax.set_ylabel("KL to no-opinion distribution")
    ax.set_title(f"{spec['name']} — held-out test questions"); ax.legend(frameon=False, fontsize=7); plots.style(ax)
    fig.tight_layout(); E.save_fig(fig, f"fig_m10_{tag}")

    del run, bases, vecs, learned
    gc.collect(); torch.cuda.empty_cache()
    return summary

## 4. Run each model

Each cell runs the whole protocol for one model and saves its tables and figure; rerunning skips models already done.

In [ ]:
# Original Qwen (Qwen2.5-1.5B-Instruct): fills in ranks 16–256 and the learned eraser under the same protocol
summaries = {}
summaries[MODELS[0]["tag"]] = run_model(MODELS[0])
summaries[MODELS[0]["tag"]].T

In [ ]:
# Current Qwen at the original's size (Qwen3.5-2B)
summaries[MODELS[1]["tag"]] = run_model(MODELS[1])
summaries[MODELS[1]["tag"]].T

In [ ]:
# Current Llama models (Llama-3.2-3B-Instruct, Llama-3.1-8B-Instruct; gated, each skipped with a message if access is missing)
for spec in MODELS[2:4]:
    summaries[spec["tag"]] = run_model(spec)
    if summaries[spec["tag"]] is not None:
        display(summaries[spec["tag"]].T)

In [ ]:
# Optional models (Qwen3.5-9B, Llama-2-7B-Chat under this protocol, Muse Glimmer 30B)
for spec in MODELS[4:]:
    if spec.get("optional") and not RUN_OPTIONAL.get(spec["tag"], False):
        print(f"skipping optional {spec['name']} (set RUN_OPTIONAL['{spec['tag']}'] = True)")
        continue
    summaries[spec["tag"]] = run_model(spec)

## 5. Across models

Test sycophancy, KL and MMLU for BES and for CAA matched to it, model by model, plus the BES rank curves side by side.

In [ ]:
# Summary table across models (test questions; MMLU in accuracy points)
summaries = {k: v for k, v in summaries.items() if v is not None}
summary = pd.concat(summaries.values(), ignore_index=True)
E.save(summary, "m10_summary")
cols = ["model", "none_syco", "no_opinion_syco"]
for key in ["svd"] + [f"k{K}" for K in LEARN_K]:
    if f"{key}_syco" in summary:
        summary[f"{key}_mmlu_drop"] = 100 * (summary.mmlu_none - summary[f"{key}_mmlu"])
        summary[f"{key}_caa_mmlu_drop"] = 100 * (summary.mmlu_none - summary[f"{key}_caa_mmlu"])
        cols += [f"{key}_desc" if key == "svd" else None, f"{key}_syco", f"{key}_kl", f"{key}_caa_kl", f"{key}_p_kl_vs_caa",
                 f"{key}_mmlu_drop", f"{key}_caa_mmlu_drop"]
summary[[c for c in cols if c]].round(3).T

In [ ]:
# Figure: BES rank curves (test sycophancy and KL against rank) for every model
fig, axes = plots.subplots(1, 2, figsize=(11, 4))
for tag in summaries:
    d = pd.read_csv(E.RESULTS / f"m10_{tag}_test.csv")
    d = d[d.kind == "rank curve"].copy()
    d["k"] = d.method.str.split().str[-1].astype(int)
    d = d.sort_values("k")
    none = pd.read_csv(E.RESULTS / f"m10_{tag}_test.csv").set_index("method").loc["none"]
    axes[0].plot(d.k, d.syco - none.syco, "-o", label=tag)
    axes[1].plot(d.k, d.kl, "-o", label=tag)
for ax, yl in zip(axes, ("change in P(user's answer) vs none", "KL to no-opinion distribution")):
    ax.set_xscale("log", base=2); ax.set_xlabel("BES rank (directions erased)"); ax.set_ylabel(yl); plots.style(ax)
axes[0].axhline(0, color="#999999", lw=0.8); axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); E.save_fig(fig, "fig_m10_rank_curves")